# Exportacion del modelo: TensorFlow.js y TFLite

El modelo entrenado sale en dos formatos, y tienen que **dar el mismo
resultado**: la app debe verse igual en el navegador y en el celular. Las
ultimas celdas lo comprueban con datos reales en vez de darlo por hecho.

## Antes de empezar

Hace falta el modelo entrenado en Drive:

    modelos/modelo_20.keras
    modelos/modelo_20_glosas.json

Y conviene tener `datasets/dataset.npz` para comparar con secuencias de
verdad. Si no esta, las celdas usan ruido y avisan.

## Por que hay un paso de "desenrollar"

La GRU bidireccional, tal como se entrena, se compila al kernel cuDNN de
NVIDIA. El grafo que sale trae las operaciones `CudnnRNNV3` y
`ReverseSequence`, y el conversor de TensorFlow.js las rechaza:

    ValueError: Unsupported Ops in the model before optimization
    ReverseSequence, CudnnRNNV3

La celda de mas abajo reconstruye la misma arquitectura con `unroll=True`,
que escribe los pasos de la GRU uno por uno en lugar de dejarlos en un bucle.
Despues de los dos MaxPool la secuencia mide 7 pasos, asi que cuesta poco.
**No cambia los pesos ni hay que reentrenar**: se copian tal cual y se
comprueba que predice identico.

In [ ]:
!pip install -q tensorflowjs

import json, os, shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from google.colab import drive

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/traduce_senas'

MODELO = f'{BASE}/modelos/modelo_20.keras'
GLOSAS = f'{BASE}/modelos/modelo_20_glosas.json'
SALIDA = f'{BASE}/export'

original = keras.models.load_model(MODELO)
glosas = json.load(open(GLOSAS, encoding='utf-8'))
print(len(glosas), 'glosas:', ', '.join(glosas))

## 1. Parchar el conversor

`tensorflowjs` no se ha actualizado al Python 3.13 / NumPy 2 de Colab: usa
`np.object`, que ya no existe, e importa un modulo de `tensorflow_hub` que
depende de `tf.compat.v1.estimator`, retirado en TF 2.20. Son dos lineas.

Si alguna vez reinicias la sesion, hay que volver a correr esta celda.

In [ ]:
import tensorflowjs, tensorflow_hub, pathlib

# np.object -> object (retirado en NumPy 2)
for ruta in pathlib.Path(tensorflowjs.__path__[0]).rglob('*.py'):
    texto = ruta.read_text(encoding='utf-8')
    if 'np.object' in texto:
        ruta.write_text(texto.replace('np.object,', 'object,').replace('np.object)', 'object)'),
                        encoding='utf-8')
        print('parchado:', ruta.name)

# el import de estimator, que ya no existe en TF 2.20
ruta = pathlib.Path(tensorflow_hub.__path__[0]) / 'estimator_export.py'
if ruta.exists() and 'tf.compat.v1.estimator' in ruta.read_text(encoding='utf-8'):
    ruta.write_text('def estimator_export(*a, **k):\n    return lambda f: f\n', encoding='utf-8')
    print('parchado:', ruta.name)

print('listo')

## 2. Desenrollar la GRU

Misma arquitectura, mismos pesos, sin cuDNN. La comprobacion de abajo tiene
que decir `misma clase: True` y una diferencia del orden de `1e-6`: si no,
algo no encajo al copiar los pesos y no hay que seguir.

In [ ]:
def build(n, seq=30, dim=128, unroll=True):
    e = keras.Input(shape=(seq, dim), name='landmarks')
    x = layers.BatchNormalization()(e)
    for f in (128, 128):
        x = layers.Conv1D(f, 5, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
        x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.GRU(96, return_sequences=False, unroll=unroll))(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    s = layers.Dense(n, activation='softmax', name='gloss')(x)
    return keras.Model(e, s, name='lsc_recognizer')

modelo = build(len(glosas))
modelo.set_weights(original.get_weights())

# Con datos reales, no con ruido: el ruido no recorre los mismos caminos
# del modelo que unas manos de verdad.
ruta_datos = f'{BASE}/datasets/dataset.npz'
if os.path.exists(ruta_datos):
    d = np.load(ruta_datos, allow_pickle=True)
    X = d['X_test'][:100].astype(np.float32)
    if len(X) == 0:
        X = d['X_train'][:100].astype(np.float32)
else:
    print('AVISO: no hay dataset.npz, se compara con ruido')
    X = np.random.randn(32, 30, 128).astype(np.float32)

base = original.predict(X, verbose=0)
salida = modelo.predict(X, verbose=0)
print(f'{len(X)} secuencias de prueba')
print('misma clase:', bool((base.argmax(1) == salida.argmax(1)).all()))
print('diferencia maxima:', f'{np.abs(base - salida).max():.2e}')

## 3. SavedModel

Los dos formatos salen de aqui. Es a proposito: si web y movil tienen que
mostrar lo mismo, es mas facil sostenerlo cuando vienen del mismo origen que
cuando son dos conversiones independientes.

Se escribe en `/content` y no en Drive porque el conversor de TensorFlow.js
falla al leer un SavedModel montado en Drive
(`OSError: SavedModel file does not exist`). Al final se copia el resultado.

In [ ]:
SM = '/content/saved_model'
TFJS = '/content/tfjs'
for d in (SM, TFJS):
    shutil.rmtree(d, ignore_errors=True)

modelo.export(SM)
print(sorted(os.listdir(SM)))

## 4. TensorFlow.js

**Sin cuantizar.** Comprimir a 8 bits ahorraria unos 200 KB, pero cambia las
salidas y rompe la paridad con el `.tflite`, que es justo lo que hay que
garantizar.

In [ ]:
!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model {SM} {TFJS} 2>&1 | tail -20

# La app espera las glosas junto al modelo, en /models/lsc/.
with open(f'{TFJS}/glosas.json', 'w', encoding='utf-8') as fh:
    json.dump(glosas, fh, ensure_ascii=False, indent=2)

!ls -lh {TFJS}

## 5. TFLite

Con la GRU desenrollada ya no hacen falta `SELECT_TF_OPS`: el modelo cabe en
los operadores basicos de TFLite, que es lo que permite correrlo en el movil
sin arrastrar medio TensorFlow.

Se deja sin cuantizar por la misma razon que arriba.

In [ ]:
conv = tf.lite.TFLiteConverter.from_saved_model(SM)
tflite = conv.convert()
RUTA_TFLITE = '/content/modelo_20.tflite'
open(RUTA_TFLITE, 'wb').write(tflite)
print(f'{len(tflite)/1e6:.2f} MB')

## 6. Dan lo mismo?

Se pasan las mismas secuencias por el modelo original y por el `.tflite`.
El criterio: **misma clase en el 100 % de los casos** y diferencia de
puntajes por debajo de `1e-4`.

Si esto falla, no copies los archivos al frontend: significa que el modelo
que veria la gente no es el que medimos.

In [ ]:
interp = tf.lite.Interpreter(model_content=tflite)
interp.allocate_tensors()
ent, sal = interp.get_input_details()[0], interp.get_output_details()[0]

filas = []
for fila in X:
    interp.set_tensor(ent['index'], fila[None].astype(np.float32))
    interp.invoke()
    filas.append(interp.get_tensor(sal['index'])[0])
filas = np.array(filas)

igual = bool((filas.argmax(1) == base.argmax(1)).all())
dif = float(np.abs(filas - base).max())
print('misma clase:', igual)
print('diferencia maxima:', f'{dif:.2e}')
print('VEREDICTO:', 'ok' if igual and dif < 1e-4 else 'REVISAR antes de publicar')

## 7. Guardar en Drive

In [ ]:
shutil.rmtree(SALIDA, ignore_errors=True)
shutil.copytree(TFJS, f'{SALIDA}/tfjs')
shutil.copy(RUTA_TFLITE, f'{SALIDA}/modelo_20.tflite')
modelo.save(f'{BASE}/modelos/modelo_20_unroll.keras')

for raiz, _, archivos in os.walk(SALIDA):
    for a in sorted(archivos):
        p = os.path.join(raiz, a)
        print(f'{os.path.getsize(p)/1024:8.0f} KB  {p.replace(SALIDA, "export")}')

## 8. Que hacer con los archivos

Descarga la carpeta `export/tfjs` de tu Drive y copia su contenido en el repo
del frontend, en:

```
frontend/public/models/lsc/
    model.json
    group1-shard1of1.bin
    glosas.json
```

Es `public/`, no `src/`: Expo publica esa carpeta en la raiz del sitio, que es
donde la app busca (`WORD_MODEL_URL` en `app/config/api.config.ts`). Los `.bin`
se llaman como diga `model.json`, asi que no hay que renombrarlos.

Si los archivos no estan, la app sigue funcionando con los 7 gestos de Google,
asi que no se rompe nada mientras tanto. Cuando si estan, el motor activo
aparece en pantalla como `modelo-lsc`.

El `.tflite` es para la app movil nativa (`react-native-fast-tflite`) y no va
en `public/`.